# PPO: Qbert (Hyperparameter Variation: lr=1e-4)

**Algorithm:** PPO  
**Environment:** Qbert (`QbertNoFrameskip-v4`)  
**Learning rate:** 1e-4 (lower than default 3e-4)  
**Parallel envs:** 8  

In [1]:
# Configuration
ALGO            = "ppo"
ENV_KEY         = "qbert"
TOTAL_TIMESTEPS = 1000000
CHECKPOINT_FREQ = 100000
LEARNING_RATE   = 1e-4
RUN_NAME        = "lr1e-4"

In [2]:
import os

import ale_py
import gymnasium as gym
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.callbacks import CheckpointCallback
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

gym.register_envs(ale_py)

ENV_IDS = {
    "qbert":      "QbertNoFrameskip-v4",
    "donkeykong": "DonkeyKongNoFrameskip-v4",
}

run_id = f"{ALGO}_{ENV_KEY}" + (f"_{RUN_NAME}" if RUN_NAME else "")
checkpoint_dir = os.path.join("checkpoints", run_id)
os.makedirs(checkpoint_dir, exist_ok=True)
os.makedirs("logs", exist_ok=True)

print(f"Run ID      : {run_id}")
print(f"Checkpoints : {checkpoint_dir}")

Run ID      : ppo_qbert_lr1e-4
Checkpoints : checkpoints/ppo_qbert_lr1e-4


In [3]:
n_envs = 8 if ALGO == "ppo" else 1
vec_env = make_atari_env(ENV_IDS[ENV_KEY], n_envs=n_envs, seed=0)
vec_env = VecFrameStack(vec_env, n_stack=4)
print(f"Environment : {ENV_IDS[ENV_KEY]}  |  n_envs: {n_envs}")

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


Environment : QbertNoFrameskip-v4  |  n_envs: 8


## Fresh Training

Checkpoints saved automatically to `checkpoints/ppo_qbert_lr1e-4/` every 100,000 steps.

In [4]:
if ALGO == "dqn":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        buffer_size=100_000, learning_starts=10_000,
        batch_size=32, train_freq=4, target_update_interval=1_000,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = DQN("CnnPolicy", vec_env, **kwargs)
elif ALGO == "ppo":
    kwargs = dict(
        verbose=1, device="cpu", tensorboard_log="logs",
        n_steps=128, batch_size=256, n_epochs=4, ent_coef=0.01, clip_range=0.1,
    )
    if LEARNING_RATE is not None:
        kwargs["learning_rate"] = LEARNING_RATE
    model = PPO("CnnPolicy", vec_env, **kwargs)

print(f"Algorithm     : {ALGO.upper()}")
print(f"Learning rate : {LEARNING_RATE}")
print(f"Device        : {model.device}")

Using cpu device
Wrapping the env in a VecTransposeImage.
Algorithm     : PPO
Learning rate : 0.0001
Device        : cpu


In [5]:
checkpoint_callback = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // n_envs, 1),
    save_path=checkpoint_dir,
    name_prefix="checkpoint",
    save_replay_buffer=False,
    save_vecnormalize=True,
)

model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    tb_log_name=run_id,
    log_interval=100,
    progress_bar=True,
    reset_num_timesteps=True,
)

final_path = os.path.join(checkpoint_dir, "final_model")
model.save(final_path)
print(f"\nTraining complete. Final model saved to: {final_path}.zip")

Logging to logs/ppo_qbert_lr1e-4_1


Output()

-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 1.39e+03      |
|    ep_rew_mean          | 270           |
| time/                   |               |
|    fps                  | 216           |
|    iterations           | 100           |
|    time_elapsed         | 473           |
|    total_timesteps      | 102400        |
| train/                  |               |
|    approx_kl            | 0.00076926034 |
|    clip_fraction        | 0.0115        |
|    clip_range           | 0.1           |
|    entropy_loss         | -1.48         |
|    explained_variance   | 0.686         |
|    learning_rate        | 0.0001        |
|    loss                 | 0.116         |
|    n_updates            | 396           |
|    policy_gradient_loss | 7.42e-05      |
|    value_loss           | 0.318         |
-------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.32e+03     |
|    ep_rew_mean          | 297          |
| time/                   |              |
|    fps                  | 216          |
|    iterations           | 200          |
|    time_elapsed         | 945          |
|    total_timesteps      | 204800       |
| train/                  |              |
|    approx_kl            | 0.0007890505 |
|    clip_fraction        | 0.0283       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.27        |
|    explained_variance   | 0.831        |
|    learning_rate        | 0.0001       |
|    loss                 | 0.0583       |
|    n_updates            | 796          |
|    policy_gradient_loss | 0.000301     |
|    value_loss           | 0.161        |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.44e+03     |
|    ep_rew_mean          | 295          |
| time/                   |              |
|    fps                  | 216          |
|    iterations           | 300          |
|    time_elapsed         | 1416         |
|    total_timesteps      | 307200       |
| train/                  |              |
|    approx_kl            | 0.0017423877 |
|    clip_fraction        | 0.0471       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.42        |
|    explained_variance   | 0.79         |
|    learning_rate        | 0.0001       |
|    loss                 | 0.057        |
|    n_updates            | 1196         |
|    policy_gradient_loss | -0.00153     |
|    value_loss           | 0.154        |
------------------------------------------


...

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.4e+03      |
|    ep_rew_mean          | 287          |
| time/                   |              |
|    fps                  | 216          |
|    iterations           | 600          |
|    time_elapsed         | 2833         |
|    total_timesteps      | 614400       |
| train/                  |              |
|    approx_kl            | 0.0024234327 |
|    clip_fraction        | 0.0378       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.54        |
|    explained_variance   | 0.868        |
|    learning_rate        | 0.0001       |
|    loss                 | 0.0324       |
|    n_updates            | 2396         |
|    policy_gradient_loss | -0.0024      |
|    value_loss           | 0.116        |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.59e+03     |
|    ep_rew_mean          | 360          |
| time/                   |              |
|    fps                  | 216          |
|    iterations           | 700          |
|    time_elapsed         | 3303         |
|    total_timesteps      | 716800       |
| train/                  |              |
|    approx_kl            | 0.0043207766 |
|    clip_fraction        | 0.0667       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.64        |
|    explained_variance   | 0.889        |
|    learning_rate        | 0.0001       |
|    loss                 | 0.0521       |
|    n_updates            | 2796         |
|    policy_gradient_loss | -0.00016     |
|    value_loss           | 0.2          |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.67e+03     |
|    ep_rew_mean          | 436          |
| time/                   |              |
|    fps                  | 217          |
|    iterations           | 800          |
|    time_elapsed         | 3774         |
|    total_timesteps      | 819200       |
| train/                  |              |
|    approx_kl            | 0.0027795571 |
|    clip_fraction        | 0.0942       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.51        |
|    explained_variance   | 0.875        |
|    learning_rate        | 0.0001       |
|    loss                 | 0.0577       |
|    n_updates            | 3196         |
|    policy_gradient_loss | -0.000406    |
|    value_loss           | 0.184        |
------------------------------------------


------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 1.79e+03     |
|    ep_rew_mean          | 471          |
| time/                   |              |
|    fps                  | 215          |
|    iterations           | 900          |
|    time_elapsed         | 4273         |
|    total_timesteps      | 921600       |
| train/                  |              |
|    approx_kl            | 0.0032897012 |
|    clip_fraction        | 0.0979       |
|    clip_range           | 0.1          |
|    entropy_loss         | -1.56        |
|    explained_variance   | 0.859        |
|    learning_rate        | 0.0001       |
|    loss                 | 0.0436       |
|    n_updates            | 3596         |
|    policy_gradient_loss | -0.00065     |
|    value_loss           | 0.129        |
------------------------------------------



Training complete. Final model saved to: checkpoints/ppo_qbert_lr1e-4/final_model.zip
